# 00 Environment Setup

Run this first in VS Code or Colab connector. It installs dependencies and confirms paths.

In [ ]:
from pathlib import Path
import os
import sys

ABS_PROJECT_ROOT = Path('/Volumes/LocalDrive1/onlyscholar-projects/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')

# If this is a Colab/remote runtime, mount Drive before searching for the project.
try:
    from google.colab import drive  # type: ignore
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception as exc:
    print('Colab Drive mount skipped:', type(exc).__name__, exc)


def find_project_root():
    candidates = []
    env_root = os.environ.get('ECOTRUST_PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path.cwd().parent, DRIVE_PROJECT_ROOT, ABS_PROJECT_ROOT])
    for base in list(candidates):
        candidates.extend(base.parents)
    seen = set()
    for candidate in candidates:
        expanded = candidate.expanduser()
        resolved = expanded.resolve() if expanded.exists() else expanded
        if str(resolved) in seen:
            continue
        seen.add(str(resolved))
        if (resolved / 'configs' / 'project.json').exists() and (resolved / 'src' / 'ecotrust').exists():
            return resolved
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT and str(PROJECT_ROOT).startswith('/content/drive') and (PROJECT_ROOT / 'configs' / 'project_colab.json').exists():
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project_colab.json'
elif PROJECT_ROOT:
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project.json'
else:
    CONFIG_PATH = None
print('Python executable:', sys.executable)
print('Python version:', sys.version)
print('Current working directory:', Path.cwd())
print('Resolved project root:', PROJECT_ROOT)
print('Config path:', CONFIG_PATH)
print('Local Mac project visible:', ABS_PROJECT_ROOT.exists())
print('Drive project visible:', DRIVE_PROJECT_ROOT.exists())
if PROJECT_ROOT:
    print('requirements-core.txt exists:', (PROJECT_ROOT / 'requirements-core.txt').exists())
    print('config exists:', CONFIG_PATH.exists())
else:
    print('No project root is visible to this kernel yet.')
if sys.version_info < (3, 9):
    raise RuntimeError('Use Python 3.9 or newer for EcoTrust-FLIDS.')


In [ ]:
import subprocess
import sys
from pathlib import Path

CORE_PACKAGES = [
    'numpy>=1.26',
    'pandas>=2.2',
    'scikit-learn>=1.4',
    'scipy>=1.12',
    'pyarrow>=15.0',
    'joblib>=1.3',
    'tqdm>=4.66',
    'matplotlib>=3.8',
    'seaborn>=0.13',
    'jupyter>=1.0',
    'ipykernel>=6.29',
]
INSTALL_OPTIONAL_ML = False  # keep False for CPU audit/preprocessing; set True before notebooks 04/05 if needed
ML_PACKAGES = ['torch>=2.2', 'shap>=0.45']


def run_pip(args, required=True):
    cmd = [sys.executable, '-m', 'pip'] + list(args)
    print('RUN:', ' '.join(str(x) for x in cmd))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-4000:])
    if result.returncode == 0:
        return True
    if '--break-system-packages' not in args:
        retry = cmd + ['--break-system-packages']
        print('Retrying with --break-system-packages because this kernel may use a system Python.')
        result = subprocess.run(retry, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(result.stdout[-4000:])
        if result.returncode == 0:
            return True
    if required:
        raise RuntimeError('Required pip install failed. See the pip output above.')
    print('Optional install failed; continuing because this stage does not need it.')
    return False

run_pip(['install', '--upgrade', 'pip'], required=False)
if PROJECT_ROOT and (PROJECT_ROOT / 'requirements-core.txt').exists():
    run_pip(['install', '-r', str(PROJECT_ROOT / 'requirements-core.txt')], required=True)
else:
    print('requirements-core.txt is not visible to this kernel. Installing core packages inline.')
    run_pip(['install', *CORE_PACKAGES], required=True)

if PROJECT_ROOT and (PROJECT_ROOT / 'pyproject.toml').exists():
    run_pip(['install', '-e', str(PROJECT_ROOT), '--no-deps'], required=True)
else:
    print('Project source is not visible, so editable install is skipped for now.')

if INSTALL_OPTIONAL_ML:
    if PROJECT_ROOT and (PROJECT_ROOT / 'requirements-ml.txt').exists():
        run_pip(['install', '-r', str(PROJECT_ROOT / 'requirements-ml.txt')], required=False)
    else:
        run_pip(['install', *ML_PACKAGES], required=False)
else:
    print('Skipped optional ML packages. Install them later before notebooks 04/05 if Torch is missing.')


In [ ]:
from pathlib import Path
import sys

if PROJECT_ROOT is None or not (PROJECT_ROOT / 'src' / 'ecotrust').exists():
    raise RuntimeError(
        'This kernel cannot access the IMPLEMENTATION source folder. '        'I have synced the code to Google Drive path: projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION. '        'In Colab, mount Drive and set ECOTRUST_PROJECT_ROOT to '        '/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION.'
    )

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
RUN_MODE = 'full'  # change to 'full' after smoke run passes
DATASETS = ['Edge-IIoTset', 'TON_IoT', 'CICIoT2023']

from ecotrust.config import load_config, ensure_project_dirs
cfg = load_config(CONFIG_PATH)
ensure_project_dirs(cfg)
print('Datasets:', list(cfg.datasets))
print('Dataset root:', cfg.dataset_root)
print('Results:', cfg.results_root)
print('Drive output:', cfg.drive_output_url)
